# IZQAR PaySim v5 portable export
This kernel converts the already sealed PaySim v5 artifact into portable XGBoost JSON plus preprocessing. It does not attach PaySim CSV, train, select a threshold, or evaluate any split. Add the output of `hoogle57/izqar-paysim-v2-sealed-final-v1` as a kernel source.


In [ ]:
from pathlib import Path
import subprocess, sys
PINNED_REVISION = 'b93248a7f80769049d154b503c58376d8ad7c42a'
root = Path('/kaggle/working/izqar')
subprocess.run(['git', 'clone', '--depth', '1', '--branch', 'feat/ml-pipelines-and-startup', 'https://github.com/Night011/izqar.git', str(root)], check=True)
actual = subprocess.check_output(['git', '-C', str(root), 'rev-parse', 'HEAD'], text=True).strip()
if actual != PINNED_REVISION:
    raise RuntimeError(f'Pinned source mismatch: expected {PINNED_REVISION}, got {actual}')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(root / 'backend/requirements-ml.txt')], check=True)
sources = list(Path('/kaggle/input').rglob('paysim_v5_pre_transaction_v2_sealed.joblib'))
if len(sources) != 1:
    raise FileNotFoundError(f'Expected exactly one sealed source joblib from the kernel input; found {len(sources)}')
source = sources[0]
if not source.with_suffix('.metrics.json').is_file():
    raise FileNotFoundError('Sealed metrics report is required beside the source artifact')
print('Converting sealed artifact only:', source)


In [ ]:
from pathlib import Path
import subprocess, sys, zipfile
backend = root / 'backend'
output_stem = Path('/kaggle/working/paysim_v5_pre_transaction_v2_sealed_portable')
bundle = Path('/kaggle/working/izqar_paysim_v5_portable_artifacts.zip')
subprocess.run([sys.executable, str(backend / 'scripts/export_portable_model.py'), '--source', str(source), '--output-stem', str(output_stem), '--bundle', str(bundle)], cwd=backend, check=True)
if not bundle.is_file():
    raise RuntimeError('Portable export bundle was not created')
with zipfile.ZipFile(bundle) as archive:
    if archive.testzip() is not None:
        raise RuntimeError('Portable export CRC failed')
    print('Portable entries:', archive.namelist())
print('Download portable output:', bundle)
